<a href="https://colab.research.google.com/github/carlosguzmans/rag-industrial/blob/main/Tarea7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!mkdir -p skills/ticket-work skills/Gestion-work skills/explicación-work skills/captura-work

Ejercicio 1

In [ ]:
#Skill 1
%%writefile skills/ticket-work/SKILL.md
---
name: ticket-work
description: Documenta un ticket de trabajo por hacer en un ticket completo con cada detalle para realizarlo.
disable-model-invocation: true
---
# Documentar Ticket

Convierte la descripción de un trabajo por hacer en un ticket completo, con todos los detalles necesarios para implementarlo.

## Proceso
1. Lee todo lo que el usuario escribió.
2. No inventes datos, consulta si falta información.
3. Rellena la plantilla con los datos suministrados.
4. Muestra el ticket al usuario.

<plantilla>

# <Título del ticket>
**Contexto:** ...
**Objetivo:** ...
**Alcance:** Incluye / No incluye
**Tareas:** 1. ... 2. ...
**Criterios de aceptación:**
- [ ] Criterio 1
**Dependencias:** ...
**Riesgos:** ...
**Prioridad:** Alta / Media / Baja

</plantilla>


Overwriting skills/ticket-work/SKILL.md


In [ ]:
%%writefile skills/Gestion-work/SKILL.md
---
name: Gestion-work
description: Implementa un ticket, genera y realiza pruebas de código
disable-model-invocation: true
---
# Documentar Ticket

Basado en un ticket previamente generado, lo convierte en código funcional y genera pruebas a partir de él.

## Proceso
1. Lee el ticket completo.
2. Si algo no está claro en el ticket, consulta al usuario, no inventes código.
3. Trabaja los criterios del ticket uno a uno:
   - Escribe una prueba que falle.
   - Escribe el código para que esta prueba pase.
   - Mejora el código sin cambiar su estructura ni su comportamiento.
4. Ejecuta todas las pruebas una vez que se hayan completado todos los criterios.
5. Entrega los resultados al usuario.

# Reglas
- Para cada criterio, realiza al menos 3 pruebas.
- No agregues código fuera de lo estipulado en el ticket.
- Usa nombres claros.
- Si una regla falla, corrígela y luego continúa.

<plantilla>

# <Título del ticket>
**Qué se hizo:** ...
**Archivos creados:** ...
**Criterios cumplidos:**
- [ ] Criterio 1
**Problemas:** ...

</plantilla>

Overwriting skills/Gestion-work/SKILL.md


In [ ]:
#Skill 3
%%writefile skills/explicación-work/SKILL.md
---
name: explicación-work
description: Mediante la generación de un HTML, explica en detalle el trabajo realizado. Úsalo cuando haya terminado la implementación y se deba explicar.
disable-model-invocation: true
---
# Documentar Ticket

Toma el proceso antes realizado y lo explica en un HTML, para que cualquier persona pueda comprender qué se hizo y por qué.

## Proceso
1. Revisa el ticket y el apartado de implementación.
2. Revisa los archivos creados.
3. Si falta información, no la inventes, pregunta al usuario.
4. Genera un archivo '.html' con las secciones de la plantilla.
5. Muestra la ruta del archivo al usuario.

#Reglas
- Solo genera un HTML.
- Todo lo que utilices debe ser de forma local, sin internet.
- Usa lenguaje común, sin tecnicismos.
- Muestra código solo cuando sea necesario para la comprensión.

<plantilla>

Secciones del HTML

1. **Título**:** nombre del trabajo realizado
2. **Resumen:** qué se hizo en un párrafo de 4 a 5 líneas.
3. **Por qué se hizo:** problema a resolver.
4. **Archivos creados:** lista enumerada de los archivos.
5. **Cómo funciona:** explicación mediante un diagrama de flujo simple.
6. **Cómo probarlo:** pasos para verificar su funcionamiento.
7. **Limitaciones:** lo que no se pudo mejorar.

</plantilla>


Overwriting skills/explicación-work/SKILL.md


In [ ]:
#Skill 4
%%writefile skills/captura-work/SKILL.md
---
name: captura-work
description: Lee los documentos que entregue el usuario y captura sus requerimientos en una lista ordenada. Úsalo cuando el usuario provea documentos en cualquier formato y pida identificar requerimientos.
disable-model-invocation: true
---
# Documentar Ticket

Lee la documentación brindada y extrae de ella los requerimientos, indicando de dónde salen.

## Proceso
1. Lee los documentos provistos, sin saltarte secciones.
2. Identifica todo lo que el documento pida o restringa.
3. Clasifica cada requerimiento en:
   - Funcional: Lo que el trabajo pide hacer
   - No funcional: calidad, rendimiento, seguridad, formato, plazos.
4. No inventes información; si algo es ambiguo o contradictorio, pregunta al usuario.
5. Muestra la lista al usuario y ajústala bajo sus requerimientos.

#Reglas
- Cada requerimiento representa una sola idea.
- Escribe cada requerimiento de manera textual como en el documento.
- Indica la fuente de donde se extrajo la información (número de página, imagen, etc.)
- Si dos documentos se contradicen, señálalo en lugar de elegir uno.
- Los requerimientos que no estén claros van en la sección de dudas, no en la tabla.

<plantilla>

# Requerimientos.

1. **Documentos leídos:** lista de documentos.

|ID | Tipo | Requerimiento | Fuente |
|xxx| Funcional| ... | Documento, pagina|

2. **Dudas:** lista de dudas o cosas que no estén claras.
- Duda 1: lo que no está claro y apartado donde aparece.

</plantilla>


Writing skills/captura-work/SKILL.md


**Ejercicio 2**

Este ejercicio se desarrollo mediante VS code. El progroma implementa un chatbot que permite realzar preguntas sobre manuales que se adjuntan en PDF.

Para lograr procesar estos PDF, se debió extraer el texto de cada página mediante la librería pypdf y posteriormente se dividió en fragmentos; esto para no saturar el buffer de Ollama y que no presente errores. Posteriormente se generan embeggings mediante El modelo **nomic-embed-text**. Los fragmentos obtenidos de cada texto se vectorizan y se envian al modelo **qwen2.5:7b**, ejecutado localmente por medio de Ollama.

Debido a que el entorno de Colab no permite una fácil integración de Ollama, para utilizarlo se debe instalar Ollama y las dependencias que se encontrarán en el archivo `requirements_industria.txt`.

In [ ]:
from pathlib import Path
import numpy as np
import ollama
from pypdf import PdfReader


modelo_chat = "qwen2.5:7b"
modelo_embedding = "nomic-embed-text"


def cortar_texto(texto, tamano=800):
    partes = []

    for i in range(0, len(texto), tamano):
        parte = texto[i:i + tamano]

        if parte.strip():
            partes.append(parte)

    return partes


def leer_manuales():
    textos = []
    fuentes = []

    carpeta_manuales = Path(__file__).parent / "manuales"

    for archivo in carpeta_manuales.glob("*.pdf"):
        print("Leyendo:", archivo.name)
        pdf = PdfReader(archivo)

        for numero, pagina in enumerate(pdf.pages, start=1):
            texto = pagina.extract_text() or ""

            for parte in cortar_texto(texto):
                textos.append(parte)
                fuentes.append(f"{archivo.name}, página {numero}")

    print("Total de fragmentos:", len(textos))
    return textos, fuentes


def crear_embeddings(textos):
    embeddings = []
    tamano_lote = 10

    for i in range(0, len(textos), tamano_lote):
        lote = textos[i:i + tamano_lote]

        print(
            f"Procesando {i + 1} de {len(textos)} fragmentos..."
        )

        respuesta = ollama.embed(
            model=modelo_embedding,
            input=lote
        )

        embeddings.extend(respuesta["embeddings"])

    return embeddings


def buscar(pregunta, textos, fuentes, embeddings):
    respuesta = ollama.embed(
        model=modelo_embedding,
        input=pregunta
    )

    vector_pregunta = respuesta["embeddings"][0]
    resultados = []

    for i in range(len(textos)):
        parecido = np.dot(
            vector_pregunta,
            embeddings[i]
        )

        resultados.append(
            (parecido, textos[i], fuentes[i])
        )

    resultados.sort(
        reverse=True,
        key=lambda resultado: resultado[0]
    )

    return resultados[:3]


def preguntar_a_qwen(pregunta, resultados):
    contexto = ""

    for resultado in resultados:
        texto = resultado[1]
        fuente = resultado[2]

        contexto += f"""
Fuente: {fuente}
{texto}
"""

    mensaje = f"""
Responde usando solamente la informacion de los manuales.

Si la respuesta no aparece en los manuales, indicalo.
Menciona el nombre del archivo y la pagina utilizada.

Informacion de los manuales:
{contexto}

Pregunta:
{pregunta}
"""

    respuesta = ollama.chat(
        model=modelo_chat,
        messages=[
            {
                "role": "user",
                "content": mensaje
            }
        ]
    )

    return respuesta["message"]["content"]


print("Leyendo manuales...")

textos, fuentes = leer_manuales()

if len(textos) == 0:
    print("No hay archivos PDF en la carpeta manuales.")
    quit()

print("Creando embeddings con Ollama...")

embeddings = crear_embeddings(textos)

print("Chatbot listo.")
print("Escribe salir para terminar.\n")


while True:
    pregunta = input("Pregunta: ")

    if pregunta.lower() == "salir":
        break

    resultados = buscar(
        pregunta,
        textos,
        fuentes,
        embeddings
    )

    respuesta = preguntar_a_qwen(
        pregunta,
        resultados
    )

    print("\nRespuesta:")
    print(respuesta)
    print()

**Ejercicio 3**


Al igual que el ejercicio 2, este se desarrolló de la misma forma, a excepción de que los PDF utilizados corresponden al curso, lo que permite preguntar sobre cualquier aspecto del contenido trabajado durante estos meses. De igual forma, los PDF's y lo necesario para su utilizacion se encontrara en `requirements_clases.txt`.

In [ ]:
from pathlib import Path
import numpy as np
import ollama
from pypdf import PdfReader


modelo_chat = "qwen2.5:7b"
modelo_embedding = "nomic-embed-text"


def cortar_texto(texto, tamano=800):
    partes = []

    for i in range(0, len(texto), tamano):
        parte = texto[i:i + tamano]

        if parte.strip():
            partes.append(parte)

    return partes


def leer_manuales():
    textos = []
    fuentes = []

    carpeta_manuales = Path(__file__).parent / "IA"

    for archivo in carpeta_manuales.glob("*.pdf"):
        print("Leyendo:", archivo.name)
        pdf = PdfReader(archivo)

        for numero, pagina in enumerate(pdf.pages, start=1):
            texto = pagina.extract_text() or ""

            for parte in cortar_texto(texto):
                textos.append(parte)
                fuentes.append(f"{archivo.name}, página {numero}")

    print("Total de fragmentos:", len(textos))
    return textos, fuentes


def crear_embeddings(textos):
    embeddings = []
    tamano_lote = 10

    for i in range(0, len(textos), tamano_lote):
        lote = textos[i:i + tamano_lote]

        print(
            f"Procesando {i + 1} de {len(textos)} fragmentos..."
        )

        respuesta = ollama.embed(
            model=modelo_embedding,
            input=lote
        )

        embeddings.extend(respuesta["embeddings"])

    return embeddings


def buscar(pregunta, textos, fuentes, embeddings):
    respuesta = ollama.embed(
        model=modelo_embedding,
        input=pregunta
    )

    vector_pregunta = respuesta["embeddings"][0]
    resultados = []

    for i in range(len(textos)):
        parecido = np.dot(
            vector_pregunta,
            embeddings[i]
        )

        resultados.append(
            (parecido, textos[i], fuentes[i])
        )

    resultados.sort(
        reverse=True,
        key=lambda resultado: resultado[0]
    )

    return resultados[:3]


def preguntar_a_qwen(pregunta, resultados):
    contexto = ""

    for resultado in resultados:
        texto = resultado[1]
        fuente = resultado[2]

        contexto += f"""
Fuente: {fuente}
{texto}
"""

    mensaje = f"""
Responde usando solamente la informacion de los manuales.

Si la respuesta no aparece en los manuales, indicalo.
Menciona el nombre del archivo y la pagina utilizada.

Informacion de los manuales:
{contexto}

Pregunta:
{pregunta}
"""

    respuesta = ollama.chat(
        model=modelo_chat,
        messages=[
            {
                "role": "user",
                "content": mensaje
            }
        ]
    )

    return respuesta["message"]["content"]


print("Leyendo manuales...")

textos, fuentes = leer_manuales()

if len(textos) == 0:
    print("No hay archivos PDF en la carpeta manuales.")
    quit()

print("Creando embeddings con Ollama...")

embeddings = crear_embeddings(textos)

print("Chatbot listo.")
print("Escribe salir para terminar.\n")


while True:
    pregunta = input("Pregunta: ")

    if pregunta.lower() == "salir":
        break

    resultados = buscar(
        pregunta,
        textos,
        fuentes,
        embeddings
    )

    respuesta = preguntar_a_qwen(
        pregunta,
        resultados
    )

    print("\nRespuesta:")
    print(respuesta)
    print()

**Ejercicio 4**


Para la ejecución de este ejercicio, se utilizó la librería **sqlite3**, mediante la cual se creó una base de datos con lo requerido en las instrucciones (Busque un repuesto (Código), Consulte en el almacén (Código), Genere una orden de trabajo (falla, equipo) y Actualice el historial de fallas), luego se implemento mediante el modelo **qwen2.5:7b** de Ollama, el agente de mantencion, con un especial enfasis en la utilizacion de Reglas, las cuales, a medida que se fue probando el agente, se fueron agregando nuevas reglas.

In [ ]:

import ollama
from pathlib import Path
from datetime import datetime
import sqlite3

base = Path(__file__).parent / "datos.db"

def base_de_datos():
    conexion = sqlite3.connect(base)
    cur = conexion.cursor()

    cur.execute("""
    CREATE TABLE IF NOT EXISTS repuestos (
        codigo TEXT PRIMARY KEY,
        nombre TEXT NOT NULL,
        equipo TEXT,
        cantidad INTEGER,
        ubicacion TEXT
    )
""")

    cur.execute("""
        CREATE TABLE IF NOT EXISTS orden (
            numero INTEGER PRIMARY KEY AUTOINCREMENT,
            equipo TEXT NOT NULL,
            falla TEXT NOT NULL,
            fecha TEXT NOT NULL,
            estado TEXT NOT NULL
        )
    """)

    cur.execute("""
            CREATE TABLE IF NOT EXISTS historial (
                numero INTEGER PRIMARY KEY AUTOINCREMENT,
                equipo TEXT NOT NULL,
                falla TEXT NOT NULL,
                fecha TEXT NOT NULL,
                orden INTEGER
            )
        """)

    conexion.commit()
    conexion.close()

def repuestos():

    conexion = sqlite3.connect(base)
    cur = conexion.cursor()
    nombres = [
        ("ABB-101", "Batería SMB", "ABB IRB140", 0  ,"B-50"),
        ("ABB-102", "Mazo de cables internos", "ABB IRB140", 5  ,"B-55"),
        ("ABB-103", "Servomotor-CA", "ABB IRB140", 8  ,"B-60"),
        ("ABB-104", "Correas dentas de transmisión", "ABB IRB140", 20 ,"B-65"),
    ]
    cur.executemany("""
        INSERT OR IGNORE INTO repuestos
        (codigo, nombre, equipo, cantidad, ubicacion)
        VALUES (?, ?, ?, ?, ?)
    """, nombres)

    conexion.commit()
    conexion.close()

def buscar_repuesto(id: str) -> str:
    """
    Busca un repuesto por su código.

    Args:
        id: Código del repuesto, por ejemplo ABB-101.
    """

    conexion = sqlite3.connect(base)
    cur = conexion.cursor()

    cur.execute("""
        SELECT codigo, nombre, equipo
        FROM repuestos
        WHERE codigo = ?
    """, (id.upper(),))

    r= cur.fetchone()
    conexion.close()

    if r is None:
        return "No se encontro el repuesto"

    return ( f"Código: {r[0]}\n"
            f"Nombre: {r[1]}\n"
            f"Equipo: {r[2]}\n"
        )

def almacen(id: str) -> str:
    """
    Consulta la cantidad y ubicación de un repuesto.

    Args:
        id: Código del repuesto, por ejemplo ABB-101.
    """
    conexion = sqlite3.connect(base)
    cur = conexion.cursor()

    cur.execute("""
        SELECT nombre, ubicacion, cantidad
        FROM repuestos
        WHERE codigo = ?
    """, (id.upper(),))
    r= cur.fetchone()
    conexion.close()

    if r is None:
        return "No se encontro el repuesto"
    return ( f"Nombre: {r[0]}"  f"Ubicacion: {r[1]}"  f"Cantidad: {r[2]}"
            )

def consultar_historial(equipo: str) -> str:
    """
    Consulta el historial de fallas de un equipo.

    Args:
        equipo: Nombre o código del equipo, por ejemplo ABB IRB140.
    """

    f = datetime.now().strftime("%A/%b/%d | %I:%M %p")
    conexion = sqlite3.connect(base)
    cur = conexion.cursor()

    cur.execute("""
        SELECT falla, fecha, orden
        FROM historial
        WHERE LOWER(equipo) = LOWER(?)
        ORDER BY numero DESC
    """, (equipo,))

    r = cur.fetchall()
    conexion.close()

    if len(r) == 0:
        return f"No hay fallas registradas para el equipo {equipo}."

    texto = f"Historial de fallas de {equipo}:\n"

    for falla, fecha, numero_orden in r:
        texto += (
            f"\nOrden: OT-{numero_orden:04d}\n"
            f"Falla: {falla}\n"
            f"Fecha: {fecha}\n"
        )

    return texto

def actualizar_historial(equipo, falla, numero_orden):
    fecha = datetime.now().strftime("%A/%b/%d | %I:%M %p")

    conexion = sqlite3.connect(base)
    cur = conexion.cursor()

    cur.execute("""
        INSERT INTO historial
        (equipo, falla, fecha, orden)
        VALUES (?, ?, ?, ?)
    """, (equipo, falla, fecha, numero_orden))

    conexion.commit()
    conexion.close()

    return "Historial actualizado correctamente."

def orden(equipo: str , falla: str) -> str:
    """
    Genera una orden de trabajo y actualiza el historial.

    Args:
        equipo: Nombre o código del equipo afectado.
        falla: Descripción de la falla encontrada.
    """
    f = datetime.now().strftime("%A/%b/%d | %I:%M %p")
    conexion = sqlite3.connect(base)
    cur = conexion.cursor()

    cur.execute("""
            INSERT INTO orden
            (equipo, fecha, falla, estado)
            VALUES (?, ?, ?, ?)
        """, (equipo, f, falla, "OK"))

    num_orden = cur.lastrowid

    conexion.commit()
    conexion.close()

    actualizar_historial(
        equipo,
        falla,
        num_orden
    )

    return (
        f"Orden N:{num_orden:04d} creada\n"
        f"Equipo: {equipo}\n"
        f"Fecha: {f}\n"
        f"Falla: {falla}\n"
        f"Estado: OK\n"
    )

def buscar(nombre: str) -> str:
    """
    Busca repuestos utilizando el nombre del componente.

    Args:
        nombre: Nombre del componente que falló, por ejemplo servomotor.
    """
    conexion = sqlite3.connect(base)
    cur = conexion.cursor()

    cur.execute("""
        SELECT codigo, nombre, equipo, cantidad, ubicacion
        FROM repuestos
        WHERE LOWER(nombre) LIKE LOWER(?)
    """, (f"%{nombre}%",))
    resultados = cur.fetchall()
    conexion.close()
    if len(resultados) == 0:
        return "No se encontraron repuestos con ese nombre."

    # Primero se crea la variable
    texto = "Repuestos encontrados:\n"

    for codigo, repuesto, equipo, cantidad, ubicacion in resultados:
        texto += (
            f"\nCódigo: {codigo}\n"
            f"Repuesto: {repuesto}\n"
            f"Equipo: {equipo}\n"
            f"Cantidad: {cantidad}\n"
            f"Ubicación: {ubicacion}\n"
        )

    return texto

modelo = "qwen2.5:7b"

def agente(pregunta):
    mensaje = [
        {
            "role": "system",
            "content": """

1. Nunca inventes códigos, cantidades, ubicaciones ni órdenes.
2. Nunca simules una consulta escribiendo frases como
   '[Consultando inventario...]'.
3. Para buscar por código debes utilizar buscar_repuesto.
4. Para buscar por nombre debes utilizar buscar.
5. Para consultar existencias mediante código debes utilizar almacen.
6. Para generar una orden debes utilizar orden.
7. Responde usando únicamente los resultados reales entregados por
   las herramientas.
8. Si no utilizaste una herramienta, no afirmes que consultaste la
   base de datos.
9. Si el usuario menciona un servomotor sin código, utiliza buscar
   con el texto 'servomotor'.
10. Para consultar fallas anteriores utiliza consultar_historial.
11. Debes mencionar el nombre del equipo al entregar su historial.
"""
        },
        {
            "role": "user",
            "content": pregunta
        }
    ]

    herramientas = [
        buscar_repuesto,
        buscar,
        almacen,
        orden,
        consultar_historial
    ]
    funciones = {
        "buscar_repuesto": buscar_repuesto,
        "buscar": buscar,
        "almacen": almacen,
        "orden": orden,
        "consultar_historial": consultar_historial
    }

    respuesta = ollama.chat(
        model = modelo,
        messages = mensaje,
        tools= herramientas,
        options={"temperature": 0}
    )

    if not respuesta.message.tool_calls:
        return (
        "No se realizó ninguna consulta en la base de datos. "
        "Indique el código o el nombre del repuesto."
    )

    resultados_herramientas = []

    for llamada in respuesta.message.tool_calls:
        nombre = llamada.function.name
        argumentos = llamada.function.arguments

        funcion = funciones.get(nombre)

        if funcion is None:
            resultado = "La herramienta solicitada no existe."
        else:
            resultado = funcion(**argumentos)

        resultados_herramientas.append(str(resultado))

    return "\n".join(resultados_herramientas)

base_de_datos()
repuestos()

while True:
    pregunta = input("\nUsuario: ")

    if pregunta.lower() == "salir":
        break

    respuesta = agente(pregunta)

    print("\nAgente:")
    print(respuesta)







**Ejercicio 5**
Suponga una aplicación usando un microcontrolador, por ejemplo un ESP32, podría usar un LLM?
que restricciones tendría.

La respuesta a esta pregunta es si, dado que actualmente existen modelos de LLM tales como **TinyLLM**, los cuales permiten correr modelos ultracompactos directamente desde el microcontrolador.

De igual forma hay que comprender los diferentes modelos de ESP32 Que se presentan actualmente, entre ellos los más adecuados para estas tareas: ESP32-S3 y el ESP32-P4, ambos modelos con soporte de instrucciones vectoriales, PSRAM externa y memoria flash aumentada de hasta 16 MB.

Las principales limitaciones que presentaría un microcontrolador como el ESP32 serían: la baja velocidad en que se genera texto, dependiendo de su memoria flash y PSRAM, además de poseer bajas capacidades cognitivas, por que su uso es principalmente para tareas muy acotadas.

Esto es hablando de correr modelos locales, pero si nos referimos a IoT, estos dispositivos se pueden utilizar como puente para grandes modelos como ChatGPT.

